week-6 Assignment

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


In [3]:
from pyspark.sql import Row


mock_data = [
    Row(product_id=1, category='Electronics', price='1200.50', old_name='Laptop', status='Completed', amount=1500, base_price=1000.0, user_id=101, region='North', priority='High'),
    Row(product_id=2, category='Furniture', price='150.00', old_name='Chair', status='Pending', amount=800, base_price=150.0, user_id=None, region='South', priority='Low'),
    Row(product_id=3, category='Electronics', price='200.00', old_name='Headphones', status='Completed', amount=1100, base_price=180.0, user_id=102, region='East', priority='High'),
    Row(product_id=4, category='Clothing', price='50.00', old_name='Shirt', status='Shipped', amount=50, base_price=45.0, user_id=103, region='West', priority='Medium'),
    Row(product_id=5, category='Appliances', price='400.00', old_name='Microwave', status='Completed', amount=1200, base_price=350.0, user_id=None, region='North', priority='High')
]


df_mock = spark.createDataFrame(mock_data)


df_mock.coalesce(1).write.mode("overwrite").csv("data/source.csv", header=True)


df_mock.coalesce(1).write.mode("overwrite").parquet("path/to/input")

print("Success! Mock CSV and Parquet files have been created in Colab.")

Success! Mock CSV and Parquet files have been created in Colab.


**Q1: Driver, Cluster Manager, and Executor RolesDriver.**

Ans

The central coordinator that runs the main() function, creates the SparkContext, and converts your code into a logical DAG (Directed Acyclic Graph) of tasks.  

 Cluster Manager: An external service (like YARN, Kubernetes, or Spark Standalone) responsible for allocating and managing the resources (memory and CPU) across the cluster.   

 Executor: Worker nodes that execute the individual tasks assigned by the Driver, store data in memory or on disk, and return the computed results back to the Driver.

**Q2: Lazy Evaluation Performance Benefits**

Ans

Spark delays execution until an Action is called, building a Lineage Graph (DAG) of all transformations in the meantime. This allows Spark's Catalyst Optimizer to analyze the entire chain of operations and optimize the execution plan—such as pushing filters down or combining steps—minimizing unnecessary data movement and passes over large datasets.

**Q3: Read CSV with Header and Schema Inference**

In [9]:
#Ans 3
df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
print("Data loaded successfully. Here is the schema:")
df.printSchema()

Data loaded successfully. Here is the schema:
root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- base_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



**Q4: CSV vs. Parquet (Storage & Performance)**

Ans

CSV is a row-based format, meaning queries must scan entire rows even if they only need a single column.   

Parquet is a columnar format, storing data column by column.  This matters for performance because analytical queries usually only require specific columns; Parquet drastically reduces I/O by skipping irrelevant columns and offers superior data compression.

Q5: Select Columns with **Condition**

In [8]:
#answer 5
electronics_df = df.select("product_id", "price").filter(df.category == 'Electronics')
print("Filtered Electronics:")
electronics_df.show(5)

Filtered Electronics:
+----------+------+
|product_id| price|
+----------+------+
|         1|1200.5|
|         3| 200.0|
+----------+------+



**Q6: Rename Column and Cast Data Type**

In [10]:
#Ans 6
df_revised = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", df["price"].cast(DoubleType()))
df_revised.printSchema()


root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- new_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- base_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



**Q7: Lineage Graph (DAG) and Fault Tolerance**

Ans

Spark records the sequence of transformations (the lineage) used to build a DataFrame from its source. If an Executor fails and a data partition is lost, Spark does not need to replicate data; it simply recomputes the lost partition from the original source using the exact steps recorded in the DAG.

Q8: Filter with AND Condition

In [11]:
print("--- Q8: Filter Completed AND > 1000 ---")
df.filter((df.status == 'Completed') & (df.amount > 1000)).show()

--- Q8: Filter Completed AND > 1000 ---
+----------+-----------+------+----------+---------+------+----------+-------+------+--------+
|product_id|   category| price|  old_name|   status|amount|base_price|user_id|region|priority|
+----------+-----------+------+----------+---------+------+----------+-------+------+--------+
|         1|Electronics|1200.5|    Laptop|Completed|  1500|    1000.0|    101| North|    High|
|         3|Electronics| 200.0|Headphones|Completed|  1100|     180.0|    102|  East|    High|
|         5| Appliances| 400.0| Microwave|Completed|  1200|     350.0|   NULL| North|    High|
+----------+-----------+------+----------+---------+------+----------+-------+------+--------+



**Q9: Predicate Pushdown in Parquet**

Ans

Predicate Pushdown is an optimization where filtering logic is pushed directly to the storage layer before data is read. Because Parquet files contain metadata (like min/max values for row groups), Spark can skip reading entire chunks of irrelevant data, drastically reducing disk I/O and the amount of data loaded into memory.

**Q10: Add a Computed Column (18% Tax)**

In [13]:
print("--- Q10: Add Final Price (18% tax) ---")
df_tax = df.withColumn("final_price", df.base_price * 1.18)
df_tax.select("product_id", "base_price", "final_price").show()

--- Q10: Add Final Price (18% tax) ---
+----------+----------+------------------+
|product_id|base_price|       final_price|
+----------+----------+------------------+
|         1|    1000.0|            1180.0|
|         2|     150.0|             177.0|
|         3|     180.0|212.39999999999998|
|         4|      45.0|53.099999999999994|
|         5|     350.0|             413.0|
+----------+----------+------------------+



**Q11: Transformations vs. Actions**

Ans

Transformations: Lazy operations that return a new DataFrame but do not trigger computation. Examples: filter(), select().  

Actions: Operations that trigger the actual execution of the DAG to compute a result and return it to the Driver or write it to storage. Examples: show(), count().

**Q12: End-to-End Pipeline (Read Parquet → Filter Nulls → Write CSV)**

In [14]:
print("--- Q12: Read Parquet, Filter Nulls, Write CSV ---")
df_parquet = spark.read.parquet("path/to/input")
df_clean = df_parquet.filter(df_parquet.user_id.isNotNull())
df_clean.show()

--- Q12: Read Parquet, Filter Nulls, Write CSV ---
+----------+-----------+-------+----------+---------+------+----------+-------+------+--------+
|product_id|   category|  price|  old_name|   status|amount|base_price|user_id|region|priority|
+----------+-----------+-------+----------+---------+------+----------+-------+------+--------+
|         1|Electronics|1200.50|    Laptop|Completed|  1500|    1000.0|    101| North|    High|
|         3|Electronics| 200.00|Headphones|Completed|  1100|     180.0|    102|  East|    High|
|         4|   Clothing|  50.00|     Shirt|  Shipped|    50|      45.0|    103|  West|  Medium|
+----------+-----------+-------+----------+---------+------+----------+-------+------+--------+



In [15]:
df_clean.write.mode("overwrite").csv("path/to/output", header=True)
print("Saved clean data to path/to/output")

Saved clean data to path/to/output


**Q13: Client Mode vs. Cluster Mode**

Ans

Client Mode: The Driver runs directly on the machine where the application was submitted. If that client machine disconnects or fails, the entire Spark job fails.  

Cluster Mode: The Driver runs as a process inside one of the worker nodes within the cluster itself, managed by the Cluster Manager. This makes the application independent of the user's submitting machine.

**Q14: Filter with OR Condition**

In [16]:
print("--- Q14: Filter North OR High Priority ---")
df.filter((df.region == 'North') | (df.priority == 'High')).show()

--- Q14: Filter North OR High Priority ---
+----------+-----------+------+----------+---------+------+----------+-------+------+--------+
|product_id|   category| price|  old_name|   status|amount|base_price|user_id|region|priority|
+----------+-----------+------+----------+---------+------+----------+-------+------+--------+
|         1|Electronics|1200.5|    Laptop|Completed|  1500|    1000.0|    101| North|    High|
|         3|Electronics| 200.0|Headphones|Completed|  1100|     180.0|    102|  East|    High|
|         5| Appliances| 400.0| Microwave|Completed|  1200|     350.0|   NULL| North|    High|
+----------+-----------+------+----------+---------+------+----------+-------+------+--------+



**Q15: .show(5) vs .collect() on Multi-Terabyte Datasets**

Ans


It is safer to use .show(5) because it only computes and retrieves the first five rows of the dataset. Calling .collect() forces Spark to send the entire multi-terabyte dataset back to the Driver's memory, which will inevitably crash the application with an OutOfMemory (OOM) error.